In [1]:
# imports
import pandas as pd
import re
import html

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Conv1D, GlobalMaxPooling1D, Dense
from sklearn.model_selection import train_test_split

## WHERE DATA FOR CNN WAS OBTAINED FROM

In [ ]:
# multiple csvs into 1
introduced = pd.read_csv('introduced.csv')
failed_one = pd.read_csv('failedone.csv')
passed_one = pd.read_csv('passedone.csv')
passed_both = pd.read_csv('passedboth.csv')
to_pres = pd.read_csv('to_president.csv')
veto = pd.read_csv('veto.csv')
law = pd.read_csv('became_law.csv')

full_bills = pd.concat([introduced, failed_one, passed_one, passed_both, to_pres, veto, law], axis=0)
rename = {'Legislation Number':'bill_id',  'Title':'bill_name',
       'Latest Tracker Stage':'all_status', 'Latest Summary':'summary'}
full_bills = full_bills.rename(columns=rename).drop(['Congress', 'URL'], axis=1)

timeline_order = [
    'Introduced',
    'Failed Senate',
    'Failed House',
    'Passed House',
    'Passed Senate',
    'Resolving Differences',
    'To President',
    'Failed to pass over veto',
    'Vetoed by President',
    'Pocket vetoed by President',
    'Became Law'
]

status_conversion = {
    'Agreed to in House': 'Passed House',
    'Agreed to in Senate': 'Passed Senate',
    'Passed over veto': 'Became Law',
    'Became Private Law': 'Became Law',
    'Pocket vetoed by President':'Vetoed by President'
}

full_bills['all_status'] = full_bills['all_status'].replace(status_conversion)
status_rank = {status: idx for idx, status in enumerate(timeline_order)}
full_bills['status_rank'] = full_bills['all_status'].map(status_rank)
full_bills_deduplicated = full_bills.sort_values(['bill_id', 'status_rank']).groupby('bill_id').last().reset_index()
full_bills_deduplicated = full_bills_deduplicated.drop('status_rank', axis=1)

print(f"original length: {len(full_bills)}")
print(f"unique bill_ids: {len(full_bills_deduplicated['bill_id'].unique())}")
print(f"unique bill_names: {len(full_bills_deduplicated['bill_name'].unique())}")

print(full_bills_deduplicated['all_status'].unique())
print(len(full_bills_deduplicated['all_status'].unique()))

rename = {'bill_id':'Legislation Number',
       'all_status':'Latest Tracker Stage', 'summary':'Latest Summary'}
full_bills_deduplicated = full_bills_deduplicated.rename(columns=rename).drop('bill_name', axis=1)

full_bills_deduplicated.to_csv("bill_sums.csv")

### Prepare Data

In [ ]:
bills = pd.read_csv("bill_details.csv")
print(len(bills))
# rename columns
rename = {'Legislation Number':'bill_id',  'Title':'bill_name',
       'Latest Tracker Stage':'status', 'Latest Summary':'summary'}
bills = bills.rename(columns=rename).drop('Congress', axis=1)

# drop bills with no summary
bills = bills.dropna(subset=['summary'])
print(len(bills))

2500
173


In [ ]:
# clean bill summaries
def clean_html_text(text):
    # remove <tags> and other HTML characters
    return html.unescape(re.sub(r'<.*?>', '', text))

bills['summary'] = bills['summary'].apply(clean_html_text)
bills.head()

,bill_id,URL,bill_name,status,summary
318,H.R. 6028,https://www.congress.gov/bill/119th-congress/h...,Legislative Branch Agencies Clarification Act,Introduced,Legislative Branch Agencies Clarification ActT...
327,H.R. 6019,https://www.congress.gov/bill/119th-congress/h...,To repeal certain provisions relating to notif...,Passed House,This bill repeals the authority for a Senator ...
372,H.R. 5974,https://www.congress.gov/bill/119th-congress/h...,Bureau of Prisons Pay Protection Act,Introduced,Bureau of Prisons Pay Protection ActThis bill ...
392,H.R. 5954,https://www.congress.gov/bill/119th-congress/h...,Beef Origin Labeling Accountability Act,Introduced,Beef Origin Labeling Accountability ActThis bi...
396,H.R. 5950,https://www.congress.gov/bill/119th-congress/h...,Keep SNAP and WIC Funded Act of 2025,Introduced,Keep SNAP and WIC Funded Act of 2025This bill ...


In [5]:
bills['status'].unique()

array(['Introduced', 'Passed House', 'Became Law',
       'Resolving Differences'], dtype=object)

In [6]:
# tokenize summaries
tokenizer = Tokenizer(num_words=10000)  # keep top 10k words
tokenizer.fit_on_texts(bills['summary'])
sequences = tokenizer.texts_to_sequences(bills['summary'])

maxlen = 500  # max number of tokens per summary
X = pad_sequences(sequences, maxlen=maxlen)

# one hot encode labels
le = LabelEncoder()
y_int = le.fit_transform(bills['status'])
y = to_categorical(y_int)

In [7]:
bills["status"].value_counts()

status
Introduced               135
Passed House              35
Became Law                 2
Resolving Differences      1
Name: count, dtype: int64

### Build Model

In [8]:
# model structure
model = Sequential([
    Embedding(input_dim=10000, output_dim=100, input_length=maxlen),
    Conv1D(filters=128, kernel_size=5, activation='relu'),
    GlobalMaxPooling1D(),
    Dense(64, activation='relu'),
    Dense(y.shape[1], activation='softmax')
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()


/opt/homebrew/Caskroom/miniforge/base/envs/tf_keras/lib/python3.10/site-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d (Conv1D)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d            │ ?                      │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)